# PiWD - PS3

Cel zajęć: zapoznanie się z narzędziami do budowy tabel przestawnych i spłaszczania danych

Do demonstracji wykorzystamy zbiór danych _graduation_rate_ zamieszczony na CEZ. Pliki danych muszą być umieszczone w tym samym folderze co notatnik.

Natomiast zadania samodzielne będziecie Państwo wykonywać na zbiorze danych _heart_disease.csv_ . W dalszej cześci notatnika będzie on objaśniony.

-----

Tabele przestawne są jedną z najistotniejszych operacji przetwarzania danych. Pozwalają na przystępne podsumowanie danych jak również na wydobywaine informacji z baz i hurtowni danych na etapie inżynierii danych projektu analitycznego.

## Przykłady

In [1]:
import pandas as pd
df = pd.read_csv("graduation_rate.csv")
df

,ACT composite score,SAT total score,parental level of education,parental income,high school gpa,college gpa,years to graduate
0,30,2094,some high school,62794,3.9,3.3,4
1,29,2022,some college,83775,3.9,3.3,4
2,27,2032,some college,75892,3.7,3.4,4
3,32,2218,associate's degree,84196,4.0,3.6,4
4,27,1958,some college,80541,3.6,3.5,4
...,...,...,...,...,...,...,...
995,29,2113,some high school,69339,3.8,3.7,3
996,26,1975,some high school,58990,3.5,3.5,3
997,31,2105,some college,38733,4.0,3.1,6
998,32,2160,high school,80222,4.0,3.6,5



| Indeks | Nazwa kolumny | Wyjaśnienie |
| :---: | :--- | :--- |
| **0** | `ACT composite score` | **Wynik z egzaminu ACT** – amerykańskiego standaryzowanego testu wstępnego na studia, oceniającego wiedzę ze szkół średnich. |
| **1** | `SAT total score` | **Całkowity wynik z egzaminu SAT** – innego bardzo popularnego w USA testu wstępnego na uczelnie wyższe, składającego się głównie z sekcji matematycznej oraz językowej. |
| **2** | `parental level of education` | **Poziom wykształcenia rodziców** |
| **3** | `parental income` | **Dochód rodziców** – roczne zarobki rodziców |
| **4** | `high school gpa` | **Średnia ocen ze szkoły średniej** (w USA najczęściej w skali od 0.0 do 4.0) |
| **5** | `college gpa` | **Średnia ocen ze studiów** |
| **6** | `years to graduate` | **Liczba lat ile student potrzebował do ukończenia studiów** |


Dla większej swobody prezentacji przedstawianych przykładów dodajmy kolumnę z zaokrąglonymi średnimi do pół oceny

In [2]:
df["hsgpa_cat"] = (df["high school gpa"]*2).round()/2

In [3]:
df["hsgpa_cat"].value_counts()

hsgpa_cat
4.0    517
3.5    409
3.0     71
2.5      3
Name: count, dtype: int64

**Przykład 1.** Stwórz tabelę przestawną ze średniej wyniku testu SAT dla każdej z par poziomów wykształcenia rodziców oraz kategorii średniej GPA z liceum.

Najpierw dokonujemy grupowania i agregacji.

In [4]:
grouped_df = df.groupby(["parental level of education","hsgpa_cat"])["SAT total score"].agg(["mean"])
grouped_df

mean
parental level of education hsgpa_cat             
associate's degree          3.0        1686.000000
                            3.5        1910.734375
                            4.0        2116.309524
bachelor's degree           3.0        1726.000000
                            3.5        1921.000000
                            4.0        2132.389610
high school                 2.5        1614.000000
                            3.0        1768.904762
                            3.5        1899.910000
                            4.0        2078.210526
master's degree             3.0        1758.000000
                            3.5        1956.300000
                            4.0        2160.762712
some college                3.0        1772.416667
                            3.5        1915.010870
                            4.0        2097.983193
some high school            2.5        1554.500000
                            3.0        1728.875000
                            3.5        1897.359649
                            4.0        2062.500000

Proszę zwracać uwagę na indeksy: wiersze tutaj są indeksowane dwoma wartościami (kolumny grupujące), mamy natomiast jedną kolumnę 'mean'. Aby sprowadzić dane do prostej tabeli wykonujemy `reset_index()`

Następnie z wyniku robimy tabelę przestawną

In [5]:
pivoted = grouped_df.reset_index().pivot(index="parental level of education", columns="hsgpa_cat", values="mean")
pivoted

hsgpa_cat,2.5,3.0,3.5,4.0
parental level of education,,,,
associate's degree,NaN,1686.000000,1910.734375,2116.309524
bachelor's degree,NaN,1726.000000,1921.000000,2132.389610
high school,1614.0,1768.904762,1899.910000,2078.210526
master's degree,NaN,1758.000000,1956.300000,2160.762712
some college,NaN,1772.416667,1915.010870,2097.983193
some high school,1554.5,1728.875000,1897.359649,2062.500000


Proszę zwracać uwagę na indeksy: 

In [6]:
print(pivoted.columns) # nagłówki są nazwanym wierszem (serią danych) - wartościami po których grupowaliśmy
print(pivoted.index) # indeks wierszy jest nazwaną serią danych - wartościami po których grupowaliśmy

Index([2.5, 3.0, 3.5, 4.0], dtype='float64', name='hsgpa_cat')
Index(['associate's degree', 'bachelor's degree', 'high school',
       'master's degree', 'some college', 'some high school'],
      dtype='str', name='parental level of education')


Proszę też zwrócić uwagę na to, że tam gdzie nie znaleziomo odpowiedniej kombinacji danych, wartość w tabeli przestwnej jest brakująca - NaN

**Przykład 2.** Proszę wykonać to samo ale inną metodą, gdzie bezpośrednio z danych jest wykonywane grupowanie, agregacja i tabela przestawna: pd.pivot_table

In [7]:
pt_df = pd.pivot_table(df, values="SAT total score", index="parental level of education", columns="hsgpa_cat", aggfunc='mean')
pt_df

hsgpa_cat,2.5,3.0,3.5,4.0
parental level of education,,,,
associate's degree,NaN,1686.000000,1910.734375,2116.309524
bachelor's degree,NaN,1726.000000,1921.000000,2132.389610
high school,1614.0,1768.904762,1899.910000,2078.210526
master's degree,NaN,1758.000000,1956.300000,2160.762712
some college,NaN,1772.416667,1915.010870,2097.983193
some high school,1554.5,1728.875000,1897.359649,2062.500000


**Przykład 3.** Dodaj jeszcze jedną kolumnę do danych z zaokrągleniem średniej ze studiów a następnie zbuduj tabelę przestawną z obiema zdyskretyzowanymi kolumnami jako indeksującymi kolumny i poziom wykształcenia indeksującym wiersze a zawartość stanowiłyby mediany.

In [8]:
df["cgpa_cat"] = (df["college gpa"]*2).round()/2
multipt_df = pd.pivot_table(df, values="SAT total score", index="parental level of education", columns=["hsgpa_cat","cgpa_cat"], aggfunc='median')
multipt_df

hsgpa_cat                       2.5             3.0                     3.5  \
cgpa_cat                        2.5     3.0     2.5     3.0     3.5     3.0   
parental level of education                                                   
associate's degree              NaN     NaN     NaN  1608.0  1725.0  1877.5   
bachelor's degree               NaN     NaN     NaN  1748.0  1704.0  1895.0   
high school                     NaN  1614.0     NaN  1736.5  1806.0  1897.0   
master's degree                 NaN     NaN     NaN  1758.0     NaN     NaN   
some college                    NaN     NaN     NaN  1773.5  1785.5  1908.5   
some high school             1631.0  1478.0  1671.5  1719.0  1769.0  1909.0   

hsgpa_cat                                       4.0                  
cgpa_cat                        3.5     4.0     3.0     3.5     4.0  
parental level of education                                          
associate's degree           1931.0     NaN  2074.0  2102.5  2215.0  
bachelor's degree            1959.5     NaN  2042.0  2120.0  2193.0  
high school                  1901.5  1890.0  2072.0  2064.0  2160.5  
master's degree              1956.0     NaN  2077.0  2118.5  2215.0  
some college                 1920.5     NaN  2049.0  2099.0  2136.5  
some high school             1902.0     NaN  2038.5  2076.5  2117.0

**Przykład 4.** Weź tabelę wynikową z przykładu 3. Sprowadź ją do postaci prostej ramki danych. Następnie dokonaj roztopienia tabeli przestawnej tak aby uzyskać 3-kolumnową ramkę z kolumnami reprezentującymi nazwy kolumny, kolumną z indeksami wierszy i kolumną z wartościami odpowiadajęcej określonej kombinacji wartości. Jest to operacja odwrotna do wykonania tabeli przestawnej.

(przydatne przy przygotowywaniu danych do wyrysowywania wykresów)

In [9]:
pt_df.reset_index().rename_axis(None, axis=1)

,parental level of education,2.5,3.0,3.5,4.0
0,associate's degree,NaN,1686.000000,1910.734375,2116.309524
1,bachelor's degree,NaN,1726.000000,1921.000000,2132.389610
2,high school,1614.0,1768.904762,1899.910000,2078.210526
3,master's degree,NaN,1758.000000,1956.300000,2160.762712
4,some college,NaN,1772.416667,1915.010870,2097.983193
5,some high school,1554.5,1728.875000,1897.359649,2062.500000


In [10]:
melted_df = pt_df.reset_index().rename_axis(None, axis=1).melt(id_vars='parental level of education', var_name="column", value_name='value')
melted_df

,parental level of education,column,value
0,associate's degree,2.5,NaN
1,bachelor's degree,2.5,NaN
2,high school,2.5,1614.000000
3,master's degree,2.5,NaN
4,some college,2.5,NaN
5,some high school,2.5,1554.500000
6,associate's degree,3.0,1686.000000
7,bachelor's degree,3.0,1726.000000
8,high school,3.0,1768.904762
9,master's degree,3.0,1758.000000


**Przykład 5.** Połącz ze sobą dwie tabele przestawne (z medianą i ze średnią) w jedną po indeksie wierszy.

In [11]:
pt_mean_df = pd.pivot_table(df, values="SAT total score", index="parental level of education", columns="hsgpa_cat", aggfunc='mean')
pt_median_df = pd.pivot_table(df, values="SAT total score", index="parental level of education", columns="hsgpa_cat", aggfunc='median')

pt_mean_df = pt_mean_df.reset_index().rename_axis(None, axis=1)
pt_median_df = pt_median_df.reset_index().rename_axis(None, axis=1)

pt_mean_df.merge(pt_median_df, on="parental level of education", suffixes=("_mean_SAT","_median_SAT"))

,parental level of education,2.5_mean_SAT,3.0_mean_SAT,3.5_mean_SAT,4.0_mean_SAT,2.5_median_SAT,3.0_median_SAT,3.5_median_SAT,4.0_median_SAT
0,associate's degree,NaN,1686.000000,1910.734375,2116.309524,NaN,1725.0,1911.5,2103.0
1,bachelor's degree,NaN,1726.000000,1921.000000,2132.389610,NaN,1726.0,1920.0,2120.0
2,high school,1614.0,1768.904762,1899.910000,2078.210526,1614.0,1776.0,1897.0,2073.5
3,master's degree,NaN,1758.000000,1956.300000,2160.762712,NaN,1758.0,1956.0,2135.0
4,some college,NaN,1772.416667,1915.010870,2097.983193,NaN,1773.5,1914.5,2087.0
5,some high school,1554.5,1728.875000,1897.359649,2062.500000,1554.5,1721.5,1905.5,2062.0


## Zadania samodzielne

Dla danych pacjentów umieszczonych w pliku _heart_disease.csv_ wykonaj poniższe polecenia.

In [12]:
df = pd.read_csv("heart_disease.csv")
df

,Gender,age,education,currentSmoker,cigsPerDay,BPMeds,prevalentStroke,prevalentHyp,diabetes,totChol,sysBP,diaBP,BMI,heartRate,glucose,Heart_ stroke
0,Male,39,postgraduate,0,0.0,0.0,no,0,0,195.0,106.0,70.0,26.97,80.0,77.0,No
1,Female,46,primaryschool,0,0.0,0.0,no,0,0,250.0,121.0,81.0,28.73,95.0,76.0,No
2,Male,48,uneducated,1,20.0,0.0,no,0,0,245.0,127.5,80.0,25.34,75.0,70.0,No
3,Female,61,graduate,1,30.0,0.0,no,1,0,225.0,150.0,95.0,28.58,65.0,103.0,yes
4,Female,46,graduate,1,23.0,0.0,no,0,0,285.0,130.0,84.0,23.10,85.0,85.0,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4233,Male,50,uneducated,1,1.0,0.0,no,1,0,313.0,179.0,92.0,25.97,66.0,86.0,yes
4234,Male,51,graduate,1,43.0,0.0,no,0,0,207.0,126.5,80.0,19.71,65.0,68.0,No
4235,Female,48,primaryschool,1,20.0,NaN,no,0,0,248.0,131.0,72.0,22.00,84.0,86.0,No
4236,Female,44,uneducated,1,15.0,0.0,no,0,0,210.0,126.5,87.0,19.16,86.0,NaN,No


Co oznaczają poszczególne kolumny
| Nazwa kolumny | objaśnienie |
|---:|:---|
|            Gender | płeć |
|               age | wiek |
|         education | poziom wykształcenia |
|     currentSmoker | czy obecnie pali? |
|        cigsPerDay | ile papierosów wypala dziennie? |
|            BPMeds | Leki na ciśnienie krwi |
|   prevalentStroke | czy przeszedł udar |
|      prevalentHyp | występowanie nadciśnienia |
|          diabetes | cukrzyca |
|           totChol | poziom całego cholesterolu |
|             sysBP | "ciśnienie górne" |
|             diaBP | "ciśnienie dolne" |
|               BMI | wskaźnik wagi/wzrostu |
|         heartRate | tętno - "uderzenia" serca na minutę |
|           glucose | poziom glukozy we krwii |
|     Heart_ stroke | czy wystąpił zawał serca |

Treść zadań

Za pomocą jednego polecenia

 1. Wykonaj dwoma metodami tabele przestawne z średnią BMI w zależności od wartości informacji o płci i poziomie wykształcenia.
 1. Wykonaj dwoma metodami tabele przestawne z średnią BMI w zależności od wartości informacji o cukrzycy i płci.
 1. Wykonaj tabelę przestawną wskazującą ile pali najwięcej osoba w grupach wynikających z płci i poziomu wykształcenia.
 1. Wykonaj tabelę przestawną wskazującą medianę BMI w grupach w zależności od wartości informacji o płci, poziomie wykształcenia posiadania cukrzycy. Organizacja kolumn/wierszy dowolna (które wartości mają trafić do naglówków kolumn a które do nazw wierszy).
 1. Dokonaj stopienia tabeli (`melt()`) z wybranymi kolumnami BMI i dwoma kolumnami ciśnienia, tak aby uzyskać tabelę z 3 kolumnami: BMI, typ ciśnienia, wartość.
 1. Wykonaj tabele przestawne wskazujące średnie ciśnienia górnego i dolnego w grupach wynikających z płci i poziomu wykształcenia. Następnie połącz te tabele w jedną nadając stosowne nazwy kolumnom. Na końcu "stop" (`melt()`) tabelę do tabeli 3-kolumnowej.
 1. Wykonaj odpowiednią tabelę przestawną dla poziomu wykształcenia i płci aby uzyskać analogiczny efekt jak w przypadku metody `crosstab()`

In [ ]:
# 1.1 Wykonaj dwoma metodami tabele przestawne z średnią BMI w zależności od wartości informacji o płci i poziomie wykształcenia.

grouped_df = df.groupby(["Gender","education"])["BMI"].agg(["mean"])
pivot = grouped_df.reset_index().pivot(index="education", columns="Gender", values="mean")
pivot

Gender,Female,Male
education,,
graduate,24.504187,25.992098
postgraduate,24.079082,26.144327
primaryschool,24.902493,25.952500
uneducated,26.760192,26.334194


In [ ]:
# 1.2 Wykonaj dwoma metodami tabele przestawne z średnią BMI w zależności od wartości informacji o płci i poziomie wykształcenia.

pt_df = pd.pivot_table(df, values="BMI", index="education", columns="Gender", aggfunc='mean')
pt_df

Gender,Female,Male
education,,
graduate,24.504187,25.992098
postgraduate,24.079082,26.144327
primaryschool,24.902493,25.952500
uneducated,26.760192,26.334194


In [18]:
# 2.1 Wykonaj dwoma metodami tabele przestawne z średnią BMI w zależności od wartości informacji o cukrzycy i płci.

grouped_df = df.groupby(["Gender","diabetes"])["BMI"].agg(["mean"])
pivot = grouped_df.reset_index().pivot(index="diabetes", columns="Gender", values="mean")
pivot

Gender,Female,Male
diabetes,,
0,25.441741,26.148429
1,28.486429,27.472549


In [19]:
# 2.2 Wykonaj dwoma metodami tabele przestawne z średnią BMI w zależności od wartości informacji o cukrzycy i płci.

pt_df = pd.pivot_table(df, values="BMI", index="diabetes", columns="Gender", aggfunc='mean')
pt_df

Gender,Female,Male
diabetes,,
0,25.441741,26.148429
1,28.486429,27.472549


In [21]:
# 3. Wykonaj tabelę przestawną wskazującą ile pali najwięcej osoba w grupach wynikających z płci i poziomu wykształcenia.

pt_df = pd.pivot_table(df, values="cigsPerDay", index="education", columns="Gender", aggfunc="max")
pt_df

Gender,Female,Male
education,,
graduate,43.0,70.0
postgraduate,40.0,60.0
primaryschool,43.0,60.0
uneducated,50.0,60.0


In [22]:
# 4. Wykonaj tabelę przestawną wskazującą medianę BMI w grupach w zależności od wartości informacji o płci, poziomie wykształcenia posiadania cukrzycy. Organizacja kolumn/wierszy dowolna (które wartości mają trafić do naglówków kolumn a które do nazw wierszy).

pt_df = pd.pivot_table(df, values="BMI", index="education", columns=["Gender", "diabetes"], aggfunc="median")
pt_df

Gender         Female           Male        
diabetes            0       1      0       1
education                                   
graduate       23.950  24.570  25.98  24.995
postgraduate   23.575  23.955  25.72  27.780
primaryschool  24.310  27.630  25.82  27.520
uneducated     25.985  29.060  26.33  27.865

In [30]:
# 5. Dokonaj stopienia tabeli (`melt()`) z wybranymi kolumnami BMI i dwoma kolumnami ciśnienia, tak aby uzyskać tabelę z 3 kolumnami: BMI, typ ciśnienia, wartość.

melted_df = df[["BMI", "sysBP", "diaBP"]].melt(id_vars="BMI", var_name="column", value_name='value')
melted_df

,BMI,column,value
0,26.97,sysBP,106.0
1,28.73,sysBP,121.0
2,25.34,sysBP,127.5
3,28.58,sysBP,150.0
4,23.10,sysBP,130.0
...,...,...,...
8471,25.97,diaBP,92.0
8472,19.71,diaBP,80.0
8473,22.00,diaBP,72.0
8474,19.16,diaBP,87.0


In [ ]:
# 6. Wykonaj tabele przestawne wskazujące średnie ciśnienia górnego i dolnego w grupach wynikających z płci i poziomu wykształcenia. Następnie połącz te tabele w jedną nadając stosowne nazwy kolumnom. Na końcu "stop" (`melt()`) tabelę do tabeli 3-kolumnowej.

pt_sysBP_df = pd.pivot_table(df, values="sysBP", index="education", columns="Gender", aggfunc="mean")
pt_diaBP_df = pd.pivot_table(df, values="diaBP", index="education", columns="Gender", aggfunc="mean")

pt_merge_df = pt_diaBP_df.merge(pt_sysBP_df, on="education", suffixes=("_mean_diaBP", "_mean_sysBP"))

melted_df = pt_merge_df.reset_index().rename_axis(None, axis=1).melt(id_vars="education", var_name="column", value_name="value")
melted_df

,education,column,value
0,graduate,Female_mean_diaBP,80.637149
1,postgraduate,Female_mean_diaBP,80.781726
2,primaryschool,Female_mean_diaBP,81.755890
3,uneducated,Female_mean_diaBP,83.860551
4,graduate,Male_mean_diaBP,84.008929
5,postgraduate,Male_mean_diaBP,83.286232
6,primaryschool,Male_mean_diaBP,83.214724
7,uneducated,Male_mean_diaBP,83.942085
8,graduate,Female_mean_sysBP,128.143629
9,postgraduate,Female_mean_sysBP,127.657360


In [ ]:
# 7. Wykonaj odpowiednią tabelę przestawną dla poziomu wykształcenia i płci aby uzyskać analogiczny efekt jak w przypadku metody `crosstab()`

# pd.crosstab(df["education"], df["Gender"])

pt_df = pd.pivot_table(df, values="age", index="education", columns="Gender", aggfunc="count")
pt_df

Gender,Female,Male
education,,
graduate,463,224
postgraduate,197,276
primaryschool,764,489
uneducated,943,777
